# Auto Library & Family Selection Preview

Preview what RMG's `'auto'` mode would select for thermo libraries, kinetics libraries,
transport libraries, seed mechanisms, and kinetics families — given an RMG input file.

Just set `input_file_path` below and run all cells.

In [1]:
input_file_path = '../examples/rmg/superminimal_auto/input.py'

In [2]:
import os

from rmgpy.rmg.main import RMG
from rmgpy.rmg.input import read_input_file
from rmgpy.data.auto_database import (
    AUTO,
    detect_chemistry,
    determine_chemistry_sets,
    determine_kinetics_families,
    expand_chemistry_sets,
    load_recommended_yml,
    merge_with_user_libraries,
    resolve_auto_kinetics_families,
    _has_pah_libs_keyword,
)

# Load the input file into an RMG object (without loading the database)
rmg = RMG()
read_input_file(os.path.expanduser(input_file_path), rmg)

print(f'Input file: {os.path.abspath(input_file_path)}')
print(f'Species:    {[spec.label or spec.molecule[0].to_smiles() for spec in rmg.initial_species]}')
print(f'Reactors:   {len(rmg.reaction_systems)}')
print(f'Solvent:    {rmg.solvent or "(none)"}')
print()
print('Current database() settings (before auto-selection):')
print(f'  thermoLibraries:    {rmg.thermo_libraries}')
print(f'  reactionLibraries:  {rmg.reaction_libraries}')
print(f'  seedMechanisms:     {rmg.seed_mechanisms}')
print(f'  transportLibraries: {rmg.transport_libraries}')
print(f'  kineticsFamilies:   {rmg.kinetics_families}')

Input file: /home/alon/Code/RMG-Py/examples/rmg/superminimal_auto/input.py
Species:    ['H2', 'O2']
Reactors:   1
Solvent:    (none)

Current database() settings (before auto-selection):
  thermoLibraries:    auto
  reactionLibraries:  auto
  seedMechanisms:     auto
  transportLibraries: auto
  kineticsFamilies:   auto


## Chemistry Detection

Analyze the input species and reactor conditions to determine what chemistry is present.

In [3]:
profile = detect_chemistry(rmg.initial_species, rmg.reaction_systems, rmg.solvent)
pah_libs_requested = _has_pah_libs_keyword(rmg)

print(f'Elements:     {"/".join(sorted(profile.elements_present))}')
print(f'Max T:        {profile.max_temperature:.0f} K')
print(f'Nitrogen:     {profile.has_nitrogen}')
print(f'Sulfur:       {profile.has_sulfur}')
print(f'Oxygen:       {profile.has_oxygen}')
print(f'Carbon:       {profile.has_carbon}')
print(f'Halogens:     {profile.has_halogens}')
print(f'Electrochem:  {profile.has_electrochem}')
print(f'Surface:      {profile.has_surface}')
print(f'Liquid:       {profile.has_liquid}')
print(f'<PAH_libs>:   {pah_libs_requested}')

Elements:     H/O
Max T:        1000 K
Nitrogen:     False
Sulfur:       False
Oxygen:       True
Carbon:       False
Halogens:     False
Electrochem:  False
Surface:      False
Liquid:       False
<PAH_libs>:   False


## Triggered Chemistry Sets & Kinetics Family Sets

Which named sets from `recommended_libraries.yml` and `recommended.py` would be activated.

In [4]:
chem_sets = determine_chemistry_sets(profile, pah_libs_requested)
family_sets = determine_kinetics_families(profile)

print('Chemistry sets (for thermo/kinetics/transport/seed libraries):')
for s in chem_sets:
    print(f'  - {s.value}')
print()
print('Kinetics family sets:')
for s in family_sets:
    print(f'  - {s.value}')

Chemistry sets (for thermo/kinetics/transport/seed libraries):
  - primary
  - oxidation

Kinetics family sets:
  - default


## What `'auto'` Would Select

The libraries and families that RMG would use if all fields were set to `'auto'`.
This always shows the auto-selected result, regardless of what the input file currently specifies.

In [5]:
recommended_data = load_recommended_yml(rmg.database_directory)
auto_thermo, auto_kinetics, auto_transport, auto_seeds = expand_chemistry_sets(
    recommended_data, chem_sets
)
auto_families = resolve_auto_kinetics_families(family_sets, rmg.database_directory)

def print_list(label, items):
    items = items or []
    print(f'\n{label} ({len(items)}):')
    for item in items:
        print(f'  {item}')

print_list('Thermo libraries', auto_thermo)
print_list('Reaction libraries', auto_kinetics)
print_list('Seed mechanisms', auto_seeds)
print_list('Transport libraries', auto_transport)
print_list('Kinetics families', auto_families)


Thermo libraries (9):
  primaryThermoLibrary
  BurkeH2O2
  Spiekermann_refining_elementary_reactions
  thermo_DFT_CCSDTF12_BAC
  DFT_QCI_thermo
  CBS_QB3_1dHR
  FFCM1(-)
  FormicAcid
  NOx2018

Reaction libraries (4):
  FormicAcid
  FFCM1(-)
  NOx2018
  2005_Senosiain_OH_C2H2

Seed mechanisms (1):
  primaryH2O2

Transport libraries (4):
  PrimaryTransportLibrary
  OneDMinN2
  NOx2018
  GRI-Mech

Kinetics families (51):
  1,2_shiftC
  H_Abstraction
  intra_NO2_ONO_conversion
  intra_H_migration
  1,3_Insertion_CO2
  Cyclic_Ether_Formation
  Disproportionation
  1,2_NH3_elimination
  1,3_Insertion_RSR
  1+2_Cycloaddition
  intra_substitutionCS_cyclization
  Cyclopentadiene_scission
  HO2_Elimination_from_PeroxyRadical
  1,3_NH3_elimination
  intra_substitutionS_isomerization
  Birad_R_Recombination
  Diels_alder_addition
  R_Recombination
  1,4_Cyclic_birad_scission
  Intra_Disproportionation
  R_Addition_COm
  Intra_R_Add_Exo_scission
  intra_substitutionS_cyclization
  Retroene
  R_Ad

## Actual Resolution of the Current Input File

Processes the input file's `database()` settings as RMG would at startup.
If the input file uses `'auto'`, `'<PAH_libs>'`, or `['!family', 'auto']`,
you'll see the resolved result. If it uses manual library lists, they pass through unchanged.

In [6]:
from rmgpy.data.auto_database import auto_select_libraries, PAH_LIBS, to_reaction_library_tuples

# Work on a fresh copy so we don't mutate the rmg object used above
import copy
rmg2 = copy.deepcopy(rmg)

# Run the same auto-selection that main.py would run
auto_select_libraries(rmg2)

# Convert reaction libraries to tuples (as main.py does before load_database)
if isinstance(rmg2.reaction_libraries, list):
    output_edge = getattr(rmg2, 'reaction_libraries_output_edge', set())
    rmg2.reaction_libraries = to_reaction_library_tuples(rmg2.reaction_libraries, output_edge)

has_auto = any(
    getattr(rmg, attr, None) == 'auto'
    or (isinstance(getattr(rmg, attr, None), list) and 'auto' in getattr(rmg, attr))
    for attr in ('thermo_libraries', 'reaction_libraries', 'seed_mechanisms',
                 'transport_libraries', 'kinetics_families')
)

if not has_auto:
    print('The input file does not use \'auto\' in any database field.')
    print('The settings below are exactly what was specified in the input file.\n')

print_list('Thermo libraries', rmg2.thermo_libraries)

rxn_lib_names = [name for name, _ in rmg2.reaction_libraries] if isinstance(rmg2.reaction_libraries, list) else rmg2.reaction_libraries
print_list('Reaction libraries', rxn_lib_names or [])

edge_libs = [name for name, flag in rmg2.reaction_libraries if flag] if isinstance(rmg2.reaction_libraries, list) else []
if edge_libs:
    print(f'\n  (output unused edge reactions for: {", ".join(edge_libs)})')

print_list('Seed mechanisms', rmg2.seed_mechanisms or [])
print_list('Transport libraries', rmg2.transport_libraries or [])

if isinstance(rmg2.kinetics_families, list):
    print_list('Kinetics families', rmg2.kinetics_families)
else:
    print(f'\nKinetics families: {rmg2.kinetics_families!r} (resolved at database load time)')


Thermo libraries (9):
  primaryThermoLibrary
  BurkeH2O2
  Spiekermann_refining_elementary_reactions
  thermo_DFT_CCSDTF12_BAC
  DFT_QCI_thermo
  CBS_QB3_1dHR
  FFCM1(-)
  FormicAcid
  NOx2018

Reaction libraries (4):
  FormicAcid
  FFCM1(-)
  NOx2018
  2005_Senosiain_OH_C2H2

Seed mechanisms (1):
  primaryH2O2

Transport libraries (4):
  PrimaryTransportLibrary
  OneDMinN2
  NOx2018
  GRI-Mech

Kinetics families (51):
  1,2_shiftC
  H_Abstraction
  intra_NO2_ONO_conversion
  intra_H_migration
  1,3_Insertion_CO2
  Cyclic_Ether_Formation
  Disproportionation
  1,2_NH3_elimination
  1,3_Insertion_RSR
  1+2_Cycloaddition
  intra_substitutionCS_cyclization
  Cyclopentadiene_scission
  HO2_Elimination_from_PeroxyRadical
  1,3_NH3_elimination
  intra_substitutionS_isomerization
  Birad_R_Recombination
  Diels_alder_addition
  R_Recombination
  1,4_Cyclic_birad_scission
  Intra_Disproportionation
  R_Addition_COm
  Intra_R_Add_Exo_scission
  intra_substitutionS_cyclization
  Retroene
  R_Ad